In [1]:
# Parameters
input = "0"  # default; papermill will override with -p input <value>

In [27]:
# Parameters
input = 0


In [28]:


# papermill visualize_gridSearch_results.ipynb out.ipynb -k bertopic_env -p input 0

# --- Parameters / CLI parsing (works in notebook and CLI) ---
# papermill visualize_gridSearch_results.ipynb out.ipynb -k bertopic_env -p input 0
# or:
# jupyter nbconvert --to notebook --execute visualize_gridSearch_results.ipynb --ExecutePreprocessor.timeout=-1 --output out.ipynb

import os
import argparse
from pathlib import Path
import pandas as pd
from bertopic import BERTopic
import plotly.io as pio

# Headless-safe Plotly renderer
pio.renderers.default = "notebook_connected" if (os.environ.get("DISPLAY") or os.environ.get("MPLBACKEND")) else "png"

# Parse CLI args
parser = argparse.ArgumentParser(description="BERTopic model loader + charts")
parser.add_argument("--input", type=str, default="0", help="level depth (e.g., 0, 1, 2, ...)")
args, _ = parser.parse_known_args()

# Allow papermill injected variable named 'input'
if "input" in globals() and isinstance(globals()["input"], (str, int)):
    level_depth = str(globals()["input"])
else:
    level_depth = args.input

print(f"[INFO] level_depth={level_depth}")

# --- Per-level layout only ---
BASE_LEVEL_DIR = Path(f"/home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_{level_depth}")
GRID_DIR       = BASE_LEVEL_DIR / "grid_search"

DF_SAMPLED_CSV = GRID_DIR / f"df_sampled_level_{level_depth}.csv"
GRID_RESULTS   = GRID_DIR / f"grid_search_results_level_{level_depth}.csv"
MODELS_DIR     = GRID_DIR / f"bertopic_models_level_{level_depth}"

# --- Validate required inputs ---
missing = [p for p in (DF_SAMPLED_CSV, GRID_RESULTS, MODELS_DIR) if not p.exists()]
if missing:
    miss_str = "\n  - ".join(str(m) for m in missing)
    raise FileNotFoundError(f"Required paths/files not found for level {level_depth}:\n  - {miss_str}")

print(f"[INFO] DF_SAMPLED_CSV = {DF_SAMPLED_CSV}")
print(f"[INFO] GRID_RESULTS   = {GRID_RESULTS}")
print(f"[INFO] MODELS_DIR     = {MODELS_DIR}")

# --- Load dataframes ---
df_sampled = pd.read_csv(DF_SAMPLED_CSV)
df_grid = pd.read_csv(GRID_RESULTS)

# --- Compute avg_score and filter (aligned with your grid search) ---
required_cols = ("silhouette", "coherence", "diversity", "hdbscan_min_cluster_size")
missing_cols = [c for c in required_cols if c not in df_grid.columns]
if missing_cols:
    raise ValueError(f"Missing columns in grid_search_results: {missing_cols}")

df_grid["avg_score"] = (df_grid["silhouette"] + df_grid["coherence"] + df_grid["diversity"]) / 3
#df_grid = df_grid[(df_grid["hdbscan_min_cluster_size"] != 50) & (df_grid["hdbscan_min_cluster_size"] != 30)]

# --- Select best rows for each metric ---
best_models = {
    "silhouette": df_grid.sort_values(by="silhouette", ascending=False).iloc[0],
    "coherence":  df_grid.sort_values(by="coherence",  ascending=False).iloc[0],
    "diversity":  df_grid.sort_values(by="diversity",  ascending=False).iloc[0],
    "avg_score":  df_grid.sort_values(by="avg_score",  ascending=False).iloc[0],
}

# Expose for later cells
print("[INFO] Data and grid loaded. Variables available:")
print(f"  MODELS_DIR = {MODELS_DIR}")
print(f"  DF_SAMPLED_CSV = {DF_SAMPLED_CSV}")
print(f"  GRID_RESULTS = {GRID_RESULTS}")
print("  best_models keys =", list(best_models.keys()))

print("df_sampled sampled")
print(df_sampled.head())


[INFO] level_depth=0
[INFO] DF_SAMPLED_CSV = /home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_0/grid_search/df_sampled_level_0.csv
[INFO] GRID_RESULTS   = /home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_0/grid_search/grid_search_results_level_0.csv
[INFO] MODELS_DIR     = /home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_0/grid_search/bertopic_models_level_0
[INFO] Data and grid loaded. Variables available:
  MODELS_DIR = /home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_0/grid_search/bertopic_models_level_0
  DF_SAMPLED_CSV = /home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_0/grid_search/df_sampled_level_0.csv
  GRID_RESULTS = /home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_0/grid_search/grid_search_results_level_0.csv
  best_models keys = ['silhouette', 'coherence', 'diversity', 'avg_score']
df_sampled sampled
                          

In [29]:
print("[INFO] Data and grid loaded. Use MODELS_DIR and best_models in the next cells.")

topic_models = {}
for key, row in best_models.items():
    suffix = (
        f"{row['model']}_"
        f"umap{row['umap_n_components']}_"
        f"umap{row['umap_n_neighbors']}_"
        f"umap{row['umap_min_dist']}_"
        f"hdbscan{row['hdbscan_min_cluster_size']}"
    )
    model_path = Path(MODELS_DIR) / suffix

    if not model_path.exists():
        print(f"[WARN] Path not found: {model_path}")
        continue

    print(f"[INFO] Loading model for best {key} from: {model_path}")
    topic_model = BERTopic.load(str(model_path))
    topic_models[key] = topic_model

    num_topics = len(set(topic_model.topics_))
    print(f"[INFO] {key}: {num_topics} topics found")

    # Headless-safe: show in notebook, else just save
    fig = topic_model.visualize_barchart(
        top_n_topics=min(80, num_topics),
        n_words=20,
        width=350,
        height=450
    )
    try:
        fig.show()
    except Exception:
        out_path = Path(f"barchart_{key}_level_{level_depth}.png")
        import plotly.io as pio
        pio.write_image(fig, out_path)
        print(f"[INFO] Saved chart to {out_path}")


[INFO] Data and grid loaded. Use MODELS_DIR and best_models in the next cells.
[INFO] Loading model for best silhouette from: /home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_0/grid_search/bertopic_models_level_0/paraphrase-MiniLM-L6-v2_umap5_umap125_umap0.0_hdbscan250
[INFO] silhouette: 3 topics found


[INFO] Loading model for best coherence from: /home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_0/grid_search/bertopic_models_level_0/paraphrase-MiniLM-L6-v2_umap3_umap125_umap0.0_hdbscan50
[INFO] coherence: 10 topics found


[INFO] Loading model for best diversity from: /home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_0/grid_search/bertopic_models_level_0/paraphrase-MiniLM-L6-v2_umap3_umap25_umap0.0_hdbscan500
[INFO] diversity: 3 topics found


[INFO] Loading model for best avg_score from: /home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_0/grid_search/bertopic_models_level_0/all-MiniLM-L6-v2_umap5_umap125_umap0.1_hdbscan100
[INFO] avg_score: 3 topics found


In [30]:
#SILHOUETTE

texts = df_sampled['text_preprocessed']
timestamps = pd.to_datetime(df_sampled['timestamp'], unit='s').dt.date
timestamps = pd.to_datetime(timestamps)

topics_over_time = topic_models['silhouette'].topics_over_time(
    docs=list(texts),
    timestamps=list(timestamps),
    nr_bins = 52,
    global_tuning = False,
    evolution_tuning = True)

topic_model.visualize_topics_over_time(
    topics_over_time,
    top_n_topics=-1)


In [31]:
topic_models['silhouette'].get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,35658,0_trump_president_like_people,"[trump, president, like, people, join, biden, ...",[president trump feel richer kamala harris cro...
1,1,286,1_trumpbuy_trumpprice_trending_chart,"[trumpbuy, trumpprice, trending, chart, events...",[trumpbuy trumpnpych holder trumpprice market ...
2,2,272,2_haley_nikki_chart_events,"[haley, nikki, chart, events, trending, market...",[nikki haley haley fqpfj holder market chart t...


In [32]:
topic_models['silhouette'].get_topic_info().set_index('Topic').loc[1,'Representative_Docs']

['trumpbuy trumpnpych holder trumpprice market chart trending events',
 'trumpbuy trumphhcjvkhg holder trumpprice market chart trending events',
 'trumpbuy trumphxyxgbc holder trumpprice market chart trending events']

In [33]:
df_sampled['topic']=topic_models['silhouette'].topics_

In [34]:
#COHERERENCE

texts = df_sampled['text_preprocessed']
timestamps = pd.to_datetime(df_sampled['timestamp'], unit='s').dt.date
timestamps = pd.to_datetime(timestamps)

topics_over_time = topic_models['coherence'].topics_over_time(
    docs=list(texts),
    timestamps=list(timestamps),
    nr_bins = 52,
    global_tuning = False,
    evolution_tuning = True)

topic_model.visualize_topics_over_time(
    topics_over_time,
    top_n_topics=-1)


In [35]:
topic_models['coherence'].get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,101,-1_chart_trending_market_position,"[chart, trending, market, position, events, sp...",[nikki haley haley fqpfj position market chart...
1,0,35215,0_trump_president_like_people,"[trump, president, like, people, join, biden, ...",[read assassination cover biden administration...
2,1,196,1_haley_nikki_holder_chart,"[haley, nikki, holder, chart, events, trending...",[nikki haley haley jnmxf holder market chart t...
3,2,176,2_trumpbuy_trumpprice_holder_trending,"[trumpbuy, trumpprice, holder, trending, chart...",[trumpbuy trumpnpych holder trumpprice market ...
4,3,159,3_blocklist_bites_automated_dust,"[blocklist, bites, automated, dust, match, ban...",[bites dust banned reason automated blocklist ...
5,4,101,4_spybox_trumponsol_alert_group,"[spybox, trumponsol, alert, group, follow, fol...",[follow alert opoaiza followed trumponsol prom...
6,5,84,5_trumpbuy_trumpprice_position_trending,"[trumpbuy, trumpprice, position, trending, cha...",[trumpbuy trumpaankzwcmz position trumpprice m...
7,6,64,6_harris_trade_trending_market,"[harris, trade, trending, market, chart, kamal...",[harris kamala position market chart trade tre...
8,7,61,7_whale_bought_value_alert,"[whale, bought, value, alert, trade, trending,...",[harris whale alert value bought holder chart ...
9,8,59,8_haley_nikki_position_chart,"[haley, nikki, position, chart, events, trendi...",[nikki haley haley bhufv position market chart...


In [36]:
topic_models['coherence'].get_topic_info().set_index('Topic').loc[1,'Representative_Docs']

['nikki haley haley jnmxf holder market chart trending events',
 'nikki haley haley ympn holder market chart trending events',
 'nikki haley haley holder market chart trending events']

In [37]:
df_sampled['topic']=topic_models['coherence'].topics_

In [38]:
#DIVERSITY

texts = df_sampled['text_preprocessed']
timestamps = pd.to_datetime(df_sampled['timestamp'], unit='s').dt.date
timestamps = pd.to_datetime(timestamps)

topics_over_time = topic_models['diversity'].topics_over_time(
    docs=list(texts),
    timestamps=list(timestamps),
    nr_bins = 52,
    global_tuning = False,
    evolution_tuning = True)

topic_model.visualize_topics_over_time(
    topics_over_time,
    top_n_topics=-1)


In [39]:
topic_models['diversity'].get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,3784,-1_trump_trending_chart_market,"[trump, trending, chart, market, events, haley...",[nikki haley haley tsfua holder market chart t...
1,0,31670,0_trump_president_like_people,"[trump, president, like, people, join, biden, ...",[president trump calls ultra left radical sooo...
2,1,762,1_pinned_migration_message_wallet,"[pinned, migration, message, wallet, token, mi...",[migration compulsory holder bought need migra...


In [40]:
topic_models['diversity'].get_topic_info().set_index('Topic').loc[1,'Representative_Docs']

['migration compulsory holder bought need migrate following pinned message instructions right project develop utilities tokens valueless migration read follow pinned message steps migration',
 'migration compulsory holder bought need migrate following pinned message instructions tokens valueless liquidate migration follow pinned message steps migrate',
 'mate proceed claim read pinned message follow pinned message instructions reward wallet minutes claiming']

In [41]:
df_sampled['topic']=topic_models['diversity'].topics_

In [42]:
#AVG_SCORE

texts = df_sampled['text_preprocessed']
timestamps = pd.to_datetime(df_sampled['timestamp'], unit='s').dt.date
timestamps = pd.to_datetime(timestamps)

topics_over_time = topic_models['avg_score'].topics_over_time(
    docs=list(texts),
    timestamps=list(timestamps),
    nr_bins = 52,
    global_tuning = False,
    evolution_tuning = True)

topic_model.visualize_topics_over_time(
    topics_over_time,
    top_n_topics=-1)


In [43]:
topic_models['avg_score'].get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,35774,0_trump_president_like_people,"[trump, president, like, people, join, biden, ...",[president trump kamala hide surrogates corrup...
1,1,272,1_haley_nikki_chart_events,"[haley, nikki, chart, events, trending, market...",[nikki haley haley vucl holder market chart tr...
2,2,170,2_blocklist_bites_automated_dust,"[blocklist, bites, automated, dust, match, ban...",[bites dust banned reason automated blocklist ...


In [44]:
topic_models['avg_score'].get_topic_info().set_index('Topic').loc[1,'Representative_Docs']

['nikki haley haley vucl holder market chart trending events',
 'nikki haley haley holder market chart trending events',
 'nikki haley haley position market chart trending events']

In [45]:
df_sampled['topic']=topic_models['avg_score'].topics_